### Atividade Busca Semântica

Nesta atividade você deve aplicar os conhecimentos sobre word embeddings e tokenização para criar um mecanismos de busca semântica. Será disponibilizado um conjunto de dados que possui perguntas médicas, em inglês. Esse conjunto de dados possui as perguntas e também as respostas.

Você deve gerar os vetores das perguntas do conjunto de dados, e permitir que o "usuário" envie a sua pergunta. Você também deve gerar o vetor da pergunta do usuário e com isso buscar a resposta ideal para o usuário. **A resposta ideal é aquela onde o vetor da pergunta do usuário é mais similar ao vetor da pergunta do conjunto de dados** Consulte o notebook "nlp2.ipynb" para verificar como realizamos esse processo

Portanto, no seu script deve ser possível escrever um texto "pergunta" e deve ser retornado a resposta adequada, isto é, a resposta associada a pergunta mais similar no conjunto de dados.

Para isso utilize o pandas e os packages do huggingface

In [ ]:
### Download do conjunto de dados

import kagglehub

# Download latest version
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")

print("Path to dataset files:", path)

/workspaces/ml-supervised-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [17]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers.tokenization_utils_base import BatchEncoding

In [ ]:
## Amostra do conjunto

df = pd.read_csv(f"{path}/medquad.csv")
df_amostra = df.sample(5000)

In [4]:
# Modelo para a língua inglesa

nome_modelo = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModel.from_pretrained(nome_modelo)

2025-09-16 01:05:31.536353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757984731.918677   31507 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757984732.026775   31507 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757984733.082719   31507 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757984733.082756   31507 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757984733.082759   31507 computation_placer.cc:177] computation placer alr

In [ ]:
## Funções para: 1) Obter os tokens; 2) Obter os embeddings

def get_tokens(pergunta: str) -> str:
    return tokenizer(pergunta, return_tensors="pt")

def get_vetores(tokens_pergunta: BatchEncoding) -> Tensor:

    with torch.no_grad():
        outputs = model(**tokens_pergunta)
        embeddings = outputs.last_hidden_state

    return embeddings

In [ ]:
## Novas colunas para os tokens e para os embeddings

df_amostra['tokens'] = df_amostra['question'].apply(lambda x: get_tokens(x))
df_amostra['vetores'] = df_amostra['tokens'].apply(lambda x: get_vetores(x))